# TP2 : General decoding

(originally written by Julien Lavauzelle. Many thanks to him for sharing his work.)

In this exercise, you will implement two algorithms solving the decoding problem.

The first, *Prange's algorithm*, is the basis of more advanced algorithms known as "information set decoding", and is used to obtain **one** solution to the problem. The second, known as *Dumer's method*, is used to get a **set** of solutions to the problem, and is therefore advantageous when the number of expected solutions is large.

We focus on the case of **binary** codes.

In [45]:
import random

## Preliminaries

In [46]:
q=2;
FF=GF(q);

**Question 1.** Write or call functions that:
- returns a vector uniformly drawn in $\mathbb{F}_2^n$
- returns the Hamming weight of a word of $\mathbb{F}_2^n$,
- returns a random subset $I \subset \{0, \dots, n-1\}$ of cadinality $k$,
- returns a random word of $\mathbb{F}_2^n$ with weight $t$,
- returns a random matrix with $k$ rows and $n$ columns over $\mathbb{F}_2$, with full rank,
- returns the submatrix $H_{J}$ of a matrix $H$ for a set $J\subset \{0, \dots, n-1\}$,
- returns the complement of the subset $I \subset \{0, \dots, n-1\}$.

*N.B.: We have shifted the indices to work between $0$ and $n-1$, as natively in Python.*

In [47]:
def random_vector(n):
    return [FF.random_element() for i in range(n)]


In [48]:
def weight(x):
    count=0
    for xi in x:
        if xi > 0:
            count+=1
    return count



In [49]:
def random_subset(n, k):
    S = Subsets([i for i in range(n)],k,submultiset=True)
    return S.random_element()

In [50]:
def random_fixed_weight(n, t):
    s=random_vector()
    while weight(s) != t :
        s=random_vector()
    return s

In [51]:
def random_binary_matrix(k, n):
    H=random_matrix(FF, k, n)
    while H.rank() != k:
        H =random_matrix(FF, k, n)
    return H

In [52]:
def submatrix(H, I):
    colonnes = H.columns()
    subH=column_matrix([colonnes(i) for i in I])
    return subH

In [54]:
def complement(I,n):
    complet = [i for i in range(n)]
    for i in I :
        if i in complet:
            complet.pop(i)
    return complet
    

## Prange's algorithm

Consider Prange's algorithm written as follows.

**Input:**  a matrix $\mathbf{H} \in \mathbb{F}_2^{(n-k) \times n}$ of full rank, an integer $t$ and a a vector $\mathbf{s}$ such that $\mathbf{s} = \mathbf{H e}$ where the Hamming weight of $\mathbf{e}$ is $t$.

**Output:**  a vector $\mathbf{x}$ of weight $t$ such that $\mathbf{H x} = \mathbf{s}$.

1. Choose randomly an information set $I$ of the code with parity check matrix $\mathbf{H}$.
1. Solve the system
$$
\left\{\begin{array}{l} \mathbf{x}_{|I} = \mathbf{0} \\ \mathbf{H x} = \mathbf{s} \end{array} \right.
$$
1. If the weight of the solution $\mathbf{x}$ is $t$, then returns $\mathbf{x}$.
2. Else, return to step 1.
    

**Question 2:** Implement this algorithm.

In [56]:
def prange(H, t, s):
    n=H.ncols()
    k = n - H.nrows()
    I = random_subset(n, k) 
    J = complement(I,n)
    HI=submatrix(H,I)
    HJ=submatrix(H,J)
    s=[1 for i in range(n-k)]
    while weight(s) != t:
        while not (HJ.is_invertible()):
            I = random_subset(n, k) 
            J=complement(I,n)
            HI=submatrix(H,I)
            HJ=submatrix(H,J)
            s=HJ.solve_right(x) 
    #en gros je dois foutre des 1 sur ceux qui sont dans J et 0 dnas les autres
    print(s)

H=random_binary_matrix(6, 10)
print(H)
prange(H, 2, [1, 0, 1, 1])
    
    
    




    
    
        

[0 1 1 1 1 1 0 1 0 1]
[1 1 1 0 1 0 1 1 0 1]
[1 0 0 0 1 0 1 1 1 0]
[0 0 1 1 1 1 1 0 0 1]
[1 0 0 1 1 0 0 0 1 1]
[1 0 0 0 1 0 1 1 0 1]


TypeError: 'list' object is not callable

**Question 3:** Test your algorithm using the functions of **Question 1**.

In [ ]:
def TEST_PRANGE():
    pass

**Question 4:** Add a counter to your algorithm pour estimate the number of iterations. Then, give the experimental complexity for the following parameters.

- $k = n/2$, $t = \lfloor 0.11 n\rfloor$ and increasing $n$ (e.g. $30$, $32$, $34$, etc.),
- $k = n/2$, $t$ between $1$ and $n/4$, with fixed $n$ (e.g. $60$ or $80$).

Compare with the expected number of iterations given in Exercise session 2. Comment the results.

*Remark: there is a strong variance in the number of iterations. For each triple $(n, k, t)$, we will consider the mean of the complexities over a dozen of samples.*

In [ ]:
#For your tests

## Exercice 2 :  Dumer's method (TD/TP)

Dumer's method is an algorithm which find a set of solutions for the decoding problem, using the birthday paradox.

### Lemma: Birthday paradox

Let $E$ be a set of cardinal $N$, and $L_1$, $L_2$ two sets of cardinality $\ell$ formed by uniformly random elements of $E$.

Then $\mathbb{E}(\# (L_1 \cap L_2))=\ell^2 / N$.

**Question 1** Observe this phenomenon using Sage. Using the function <code>random_subset(N,l)</code>, build two random sets of $\{1,\dots,N\}$ and count the number of collisions. Do it <code>NbIt</code>$\geq 100$ times and compute the mean of the number of collisions. Compare this value with the expected one.

In [ ]:
NbIt=100
#Some instances of values for N and l.
N=16
l=4
for it in range(NbIt): 
    pass



Now let's go back to decoding problem.

In this exercise, we suppose that the length $n$ is even. Let $J_1 \cup J_2$ be a partition of $\{0, \dots, n-1\}$ in two subsets of the *same* cardinality. Given a parity check matrix $\mathbf{H} \in \mathbf{F}_2^{(n-k) \times n}$, we denote by $\mathbf{H}_i = \mathbf{H}_{|J_i}$.

For any $\mathbf{s} \in \mathbb{F}_2^{n-k}$, we define :
\\[
    \begin{array}{l}
    L_1 := \{  \mathbf{H_1 x_1} \mid \mathbf{x_1} \in \mathbb{F}_2^{n/2} \text{ and } |\mathbf{x_1}| = t/2 \} \subseteq \mathbf{F}_2^{n-k} \\
    L_2 := \{  \mathbf{s} - \mathbf{H_2 x_2} \mid \mathbf{x_2} \in \mathbb{F}_2^{n/2} \text{ and } |\mathbf{x_2}| = t/2 \} \subseteq \mathbf{F}_2^{n-k}  
    \end{array}
\\]


**Question 2**
Justify that these lists have cardinality at most $\ell=\binom{n/2}{t/2}$ elements.

In [ ]:
#Foe your answer

The idea is to look for collisions between the lists $L_1$ et $L_2$ : if $\mathbf{H_1 x_1} = \mathbf{s} - \mathbf{H_2 x_2}$ and if $\mathbf{x_1}$ and $\mathbf{x_2}$ have weight $w/2$, then the vector $\mathbf{x} = (\mathbf{x_1}, \mathbf{x_2})$ is a solution of the syndrom decoding problem of weight $w$.

Of course, the existence of such a collision depends on how well the sets $J_1$ et $J_2$ 'distribute' evenly the error. As for Prange's algorithm, we repeat by drawing randomly $J_1$, $J_2$ until we find a solution.  

### Dumer's method

**Entrée :** a matrix $\mathbf{H} \in \mathbb{F}_2^{n-k}$, an integer $w$  and a syndrome $\mathbf{s} = \mathbf{H e}$ where $\mathbf{e}$ has weight $t$.

**Sortie :** a set of vectors $\mathbf{t}$ of weight $w$ such that $\mathbf{H x} = \mathbf{s}$.

1. Pick randomly a partition $J_1 \cup J_2$ of $\{0, \dots, n-1\}$.
2. Create the **dictionnaries** associated to the lists $L_1$, $L_2$.
3. Compute the set of collisions of preimages $\{ (\mathbf{x_1}, \mathbf{x_2}) \}$.
4. If this set is empty, go back to 1.
5. Else, return the concatenated vectors $\{ (\mathbf{x} = \mathbf{x_1}, \mathbf{x_2}) \} \subseteq \mathbb{F}_2^n$ of the the collisions.

**Question 3** Implement Dumer's method. You can use previous functions.

In [ ]:
def dumer(H, w, s):
    pass

**Question 4** Try it on small examples (e.g. $n = 20$, $k = 10$, $t = 8$).

In [ ]:
def TEST_DUMER():
    pass

**Question 5** Using Question 1 and playing on sets of parameters $(n,k,t)$ with $t$ large enough, observe that the expected number of solutions returned by <code>dumer</code> is $\binom{n/2}{t/2}2^{k-n}$.